# 🧪 PoC: Kinetopus Synthetic Market Generator (Ecuaciones de BTC)
**Concepto:** Extraer la física matemática (ODEs gobernantes) de un activo real altamente técnico como **BTC-USD** con SINDy, y luego utilizar el módulo estocástico de **Euler-Maruyama** de Kinetopus para proyectar un horizonte extremadamente largo (ej. 5.000 velas) y evaluar si las trayectorias sintéticas generadas son verosímiles a "ojo" a un mercado real.

In [5]:
# Activar autoreload para recargar en caliente cambios en módulos externos
%load_ext autoreload
%autoreload 2

import sys
import os

# Resolución dinámica de la raíz del proyecto (evita ModuleNotFoundError en Jupyter)
current_dir = os.path.abspath(os.getcwd())
while current_dir != os.path.dirname(current_dir):
    if 'src' in os.listdir(current_dir):
        if current_dir not in sys.path:
            sys.path.append(current_dir)
        break
    current_dir = os.path.dirname(current_dir)

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import UnivariateSpline

# Kinetopus Core
from src.ui.market_loader import MarketLoader
from src.quant_engine.blender import ContinuousBlender
from src.quant_engine.physics import PhysicsDiscoverer
from src.quant_engine.nervous import RegimeShiftDetector


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Descarga de Datos de Referencia (BTC) y Suavizado Topológico
Descargaremos las últimas 300 velas diarias de BTC para extraer la inercia física de su micro-estructura. Luego utilizaremos el `ContinuousBlender` de Kinetopus para suavizar las variedades de precio y volumen.

In [6]:
# 1. Bajar datos de BTC
df = yf.download('BTC-USD', period='1y', interval='1d')
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

# Asegurar cast a float64 en el índice temporal
t = np.arange(len(df), dtype=np.float64)

# Preparamos el tensor sanitizado (Log Returns y Volumen Z-Score)
log_returns, vol, raw_price, dt_val = MarketLoader.prepare_quant_input(df, disable_norm=False, disable_returns=False)

# Normalización Z-score de Volumen manual para alimentar las derivadas
mu_v = np.mean(vol)
sigma_v = np.std(vol) if np.std(vol) > 1e-8 else 1.0
vol_z = (vol - mu_v) / sigma_v

# 2. Suavizado Topológico continuo (Capa 2: Splines de Memoria Líquida)
blender = ContinuousBlender(tolerance=0.0025)
telemetry_r = blender.fit(t, log_returns, dominant_periods=np.array([]), feature_idx=0)
telemetry_v = blender.fit(t, vol_z, dominant_periods=np.array([]), feature_idx=1)

smooth_r, r_dot, r_dot2 = blender.compute_continuous(0, t)
v_smooth, v_dot, v_dot2 = blender.compute_continuous(1, t)

print("Preprocesamiento y variedades continuas asimiladas de forma vectorizada!")


[*********************100%***********************]  1 of 1 completed

Preprocesamiento y variedades continuas asimiladas de forma vectorizada!


## 2. Descubrimiento SINDy y Simulación de Euler-Maruyama
Usamos la clase `PhysicsDiscoverer` nativa del motor de Kinetopus para calibrar y ajustar el modelo. Luego simulamos caminos sintéticos de 5.000 días inyectando shocks estocásticos (ruido de Wiener) de acuerdo con los residuos históricos reales del activo.

In [7]:
# 3. Descubrimiento Físico con SINDy y Consenso Topológico
import importlib
import src.quant_engine.physics
importlib.reload(src.quant_engine.physics)
from src.quant_engine.physics import PhysicsDiscoverer

horizonte_sintetico = 1000
trayectorias_montecarlo = 500  # <--- Modifica aquí (ej. 5000 o 10000) para mayor estabilidad de la mediana

discoverer = PhysicsDiscoverer(poly_degree=1)
x_matrix = np.column_stack((smooth_r, v_smooth))
x_dot_matrix = np.column_stack((r_dot, v_dot))

# NUEVO: Detector CUSUM para usar la física del régimen actual (más coherente)
# Usamos los parámetros por defecto de la app para evitar micro-regímenes inestables
detector = RegimeShiftDetector(threshold=5.0, drift=1.0)
cusum_report = detector.detect(log_returns, smooth_r)
shift_idx = cusum_report['shift_indices']

boundaries = [0] + shift_idx + [len(t)]
start_idx = 0
# Buscar el último régimen que tenga al menos 15 velas para evitar over-fitting extremo
for i_b in range(len(boundaries) - 1):
    s_idx = boundaries[i_b]
    e_idx = boundaries[i_b+1]
    if e_idx - s_idx >= 15:
        start_idx = s_idx

t_slice = t[start_idx:]
x_slice = x_matrix[start_idx:]
x_dot_slice = x_dot_matrix[start_idx:]

# Recalcular residuos para el régimen activo
sigma_res_r = float(np.std(log_returns[start_idx:] - smooth_r[start_idx:]))
sigma_res_v = float(np.std(vol_z[start_idx:] - v_smooth[start_idx:]))

print(f"Iniciando extracción física y Monte Carlo ({trayectorias_montecarlo} trayectorias x {horizonte_sintetico} pasos)...")
# Ejecutar descubrimiento físico (SINDy para el régimen activo) y Monte Carlo nativo integrado
report = discoverer.extract_equations(
    t=t_slice, x=x_slice, x_dot=x_dot_slice, dt=dt_val, 
    horizon_steps=horizonte_sintetico,
    sigma_res_r=sigma_res_r, sigma_res_v=sigma_res_v,
    last_price=raw_price[-1], disable_norm=False, disable_returns=False,
    mc_paths=trayectorias_montecarlo
)

print("Ecuaciones del Atractor BTC descubiertas:")
print(f"dr/dt = {report['equations'][0]}")
print(f"dV/dt = {report['equations'][1]}")
print(f"R2 Score: {report['score']:.4f}")

# Extraer el Consenso Topológico (Mediana y Bandas de Confianza)
p_price = report['prediction']['price_percentiles']
if len(p_price) == 5:
    p5, p25, p50, p75, p95 = p_price
    print(f"Simulación de Consenso completada con éxito. Longitud proyectada: {len(p5)}")
else:
    print("Física inestable detectada, cayendo a extrapolación determinística desnuda.")
    p50 = report['prediction']['det_price_path']
    p5 = p25 = p75 = p95 = p50


Iniciando extracción física y Monte Carlo (500 trayectorias x 1000 pasos)...


TypeError: PhysicsDiscoverer.extract_equations() got an unexpected keyword argument 'mc_paths'

## 3. Inspección Visual (El Test de Verosimilitud)
Graficaremos los caminos estocásticos generados. Esto nos permite evaluar visualmente si los patrones (tendencias, micro-reversiones, autocorrelaciones) imitan el comportamiento caótico y fractal de un activo de trading real.

In [ ]:
# Generamos el eje de tiempo sintético
t_sintetico = np.arange(len(p50)) + len(t)  # Desplazado al futuro

fig = go.Figure()

# Plotting del Histórico Real (Últimas 100 velas) para contexto
t_history = t[-100:]
p_history = raw_price[-100:]
fig.add_trace(go.Scatter(
    x=t_history, y=p_history,
    mode='lines',
    name='Histórico Real',
    line=dict(color='white', width=2)
))

# Atractor Principal (Mediana - Consenso Topológico)
fig.add_trace(go.Scatter(
    x=t_sintetico, y=p50,
    mode='lines',
    name='Mediana (Consenso Topológico)',
    line=dict(color='#ff00ff', width=3)
))

fig.update_layout(
    title=f"Consenso Topológico Kinetopus (Monte Carlo {trayectorias_montecarlo} trayectorias - {horizonte_sintetico} velas)",
    template="plotly_dark",
    height=700,
    xaxis_title="Tiempo (t)",
    yaxis_title="Precio Proyectado",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()


## 4. Exportación de Datos Sintéticos para el Motor
A continuación, empaquetamos y exportamos los caminos sintéticos generados en un formato CSV de alta fidelidad, asegurando la compatibilidad absoluta con las 5 capas del motor Kinetopus (OHLC, volumen dinámico estocástico y alineación temporal).

In [ ]:
# ==============================================================================
# 4. Generación y Exportación Rígida de Datos Sintéticos para Kinetopus App
# ==============================================================================
import os
import pandas as pd
import numpy as np

# Definir la cantidad de registros sintéticos solicitados
num_datos = 2000

# Validar que tengamos suficientes datos simulados en p50
if 'p50' in locals() and len(p50) >= num_datos:
    print(f"Preparando la exportación de {num_datos} velas sintéticas coherentes...")
    
    # 1. Crear carpeta destino en la raíz del proyecto
    output_dir = os.path.join("..", "data_sintetica")
    os.makedirs(output_dir, exist_ok=True)
    csv_filename = "btc_sintetico_2000.csv"
    output_path = os.path.join(output_dir, csv_filename)
    
    # 2. Generación temporal continua
    if isinstance(df.index, pd.DatetimeIndex):
        start_date = df.index[-1] + pd.Timedelta(days=1)
        fechas = pd.date_range(start=start_date, periods=num_datos, freq="D")
    else:
        fechas = np.arange(num_datos) + len(df)
        
    # 3. Modelado de Precios OHLC Estocásticamente Coherentes (Caja Blanca)
    close_sintetico = p50[:num_datos]
    
    # Simular una volatilidad intradiaria coherente (0.8% aprox.)
    volatilidad_velas = 0.008
    open_sintetico = close_sintetico * (1.0 + np.random.normal(0, volatilidad_velas * 0.3, size=num_datos))
    high_sintetico = np.maximum(open_sintetico, close_sintetico) * (1.0 + np.abs(np.random.normal(0, volatilidad_velas * 0.5, size=num_datos)))
    low_sintetico = np.minimum(open_sintetico, close_sintetico) * (1.0 - np.abs(np.random.normal(0, volatilidad_velas * 0.5, size=num_datos)))
    
    # 4. Modelado de Volumen Dinámico Verosímil (Z-Score Friendly)
    # Extraemos la media y desviación del volumen real del histórico
    mean_v = df['Volume'].mean() if 'Volume' in df.columns else 1000.0
    std_v = df['Volume'].std() if 'Volume' in df.columns else 150.0
    # Simulación estocástica truncada a > 100.0 para evitar errores matemáticos en el sensor FFT
    volume_sintetico = np.maximum(100.0, np.random.normal(mean_v, std_v * 0.20, size=num_datos))
    
    # 5. Consolidación del DataFrame Sintético Premium
    df_sintetico = pd.DataFrame({
        'Open': open_sintetico,
        'High': high_sintetico,
        'Low': low_sintetico,
        'Close': close_sintetico,
        'Volume': volume_sintetico
    }, index=fechas)
    
    df_sintetico.index.name = 'Date'
    
    # 6. Guardar archivo CSV
    df_sintetico.to_csv(output_path)
    
    print("\n" + "="*80)
    print(" 🚀 ¡DATOS SINTÉTICOS EXPORTADOS CORRECTAMENTE PARA EL MOTOR QUANT!")
    print("="*80)
    print(f"► Ruta de Destino:  {os.path.abspath(output_path)}")
    print(f"► Total Registros:  {len(df_sintetico)} velas diarias sintéticas.")
    print("► Estructura de Columnas: ['Open', 'High', 'Low', 'Close', 'Volume']")
    print("\n✨ Especificaciones de Ingesta y Verosimilitud:")
    print("  1. Coherencia OHLC: Las velas sintéticas tienen una relación High >= Open/Close >= Low matemática.")
    print("  2. Volumen Dinámico: Simulado con la media del BTC real y oscilación gaussiana controlada.")
    print("     Esto evita señales vacías de volumen neutro que reducen la riqueza frecuencial del Sensor (FFT).")
    print("  3. Continuidad Temporal: El índice de fechas comienza exactamente al día siguiente del BTC real.")
    print("="*80)
else:
    print("Error: No se encontró la simulación p50 en las variables locales. Ejecuta las celdas previas primero.")


Error: No se encontró la simulación p50 en las variables locales. Ejecuta las celdas previas primero.
